# Branch-Level Demand Forecasting Engine

Forecasts demand per **(branch, P/N)** instead of collapsing to the national "ALL" rollup, using 8 methods (MA, WMA, EWMA, Linear Regression, Polynomial Regression degree 2/3, SES, DES) and picks the best method per part by backtest MAE.

**Multi-agency:** the engine auto-detects an `Agc` column if one is present and carries it straight through to the output, so a single consolidated export covering several agencies (e.g. a `pmovdcE` file with an "All" sheet spanning agc 06, 07, 08, 10, ...) is forecast in one pass -- branch, agency, and P/N together form the row key, so the same P/N in the same branch under two different agencies is kept as two separate rows.

**Input:** you set `INPUT_FILE`, `SHEET_NAME`, and `SKIP_ROWS` to match your export (see Section 1). Leave `SHEET_NAME`/`SKIP_ROWS` as `None` and they're auto-detected instead, so small format changes won't break it.

**Performance:** every model is vectorized with numpy across all rows at once (no per-row Python loops, no per-row sklearn/statsmodels calls), which is what makes branch-level (~30k+ rows) practical to run in seconds instead of minutes.


In [1]:
import glob
import os
import re
import time

import numpy as np
import pandas as pd

## 1. File loading

Give it the sheet name and the number of rows to skip before the header row -- the same two things you'd set in a plain `pd.read_excel(file, sheet_name=..., skiprows=...)` call. Set either one to `None` (the default) to have it auto-detected instead, which is handy if you don't know the layout yet or it shifts slightly between exports.


In [2]:
def find_data_sheet(path):
    """Picks the sheet most likely to hold the movement data.
    Prefers a sheet whose name contains 'pmovdc'; falls back to the first sheet.
    Only used when sheet_name isn't given explicitly to load_movement_data().
    """
    xl = pd.ExcelFile(path)
    for name in xl.sheet_names:
        if 'pmovdc' in name.lower():
            return name
    return xl.sheet_names[0]


def find_header_row(path, sheet_name, max_scan=15):
    """Scans the first `max_scan` rows to find the one that looks like the
    real header row (contains a P/N column and a branch column), regardless
    of how many metadata rows precede it. Only used when skiprows isn't
    given explicitly to load_movement_data().
    """
    preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_scan)
    for i, row in preview.iterrows():
        vals = [str(v).strip().lower() for v in row.values]
        has_pn = any(v in ('p/n', 'pn', 'part number') for v in vals)
        has_branch = any(v in ('brc', 'branch', 'br') for v in vals)
        if has_pn and has_branch:
            return i
    raise ValueError(
        f"Could not auto-detect the header row in the first {max_scan} rows of "
        f"sheet '{sheet_name}'. Open the file and check where the real column "
        f"headers (P/N, Brc, etc.) start, then pass skiprows explicitly."
    )


def load_movement_data(file_path, sheet_name=None, skiprows=None):
    """Loads the file you point it at.

    sheet_name / skiprows: pass these explicitly once you know them (e.g. a
    multi-agency export like "pmovdcE_CNH_2_Jul_26.xlsx" -> sheet_name="All",
    skiprows=4) for a fast, deterministic load. Leave either as None and it's
    auto-detected instead, so the notebook still works if the export layout
    shifts slightly (extra sheets, an inserted metadata row, etc.).
    """
    sheet = sheet_name if sheet_name is not None else find_data_sheet(file_path)
    rows_to_skip = skiprows if skiprows is not None else find_header_row(file_path, sheet)
    df = pd.read_excel(file_path, sheet_name=sheet, skiprows=rows_to_skip)
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]
    print(f"Loaded: {file_path}")
    print(f"  sheet: '{sheet}', skiprows: {rows_to_skip}, rows: {len(df)}")
    return df


## 2. Vectorized forecasting models

Each function takes a `(n_rows, 12)` actual-demand array and returns a `(n_rows, 13)` forecast series (12 in-sample fitted points + 1 forward forecast), computed for **all rows at once**.

In [3]:
def batch_ma(clipped_15):
    """Simple moving average, 3-point window, over a 15-length input -> 13 outputs."""
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)  # (n,13,3)
    return windows.mean(axis=2)


def batch_wma(clipped_15, d_last, step=0.05):
    """Weighted moving average with weight-grid search (w3 > w2 > w1, sum=1).
    Loops only over the ~100-150 valid weight combos, not over rows.
    """
    from numpy.lib.stride_tricks import sliding_window_view
    windows = sliding_window_view(clipped_15, window_shape=3, axis=1)
    n = clipped_15.shape[0]

    combos = []
    for w1 in np.round(np.arange(0.15, 0.81, step), 2):
        for w2 in np.round(np.arange(0.25, 0.86 - w1, step), 2):
            w3 = round(1 - (w1 + w2), 2)
            if w3 > w2 > w1:
                combos.append((w1, w2, w3))

    best_rmse = np.full(n, np.inf)
    best_series = np.zeros((n, 13))
    best_weights = np.zeros((n, 3))

    for w1, w2, w3 in combos:
        forecast = (windows[:, :, 0] * w1 + windows[:, :, 1] * w2 + windows[:, :, 2] * w3) / (w1 + w2 + w3)
        rmse = np.abs(d_last - forecast[:, -1])
        better = rmse < best_rmse
        best_rmse = np.where(better, rmse, best_rmse)
        best_series[better] = forecast[better]
        best_weights[better] = [w1, w2, w3]

    return best_series, best_weights

In [4]:
def batch_ewma(clipped_12, alpha=0.4):
    n, T = clipped_12.shape
    ewma = np.zeros((n, T))
    ewma[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        ewma[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * ewma[:, t - 1]
    forward = ewma[:, -1]
    return np.concatenate([ewma, forward[:, None]], axis=1)

In [5]:
def _batch_poly(clipped_12, degree):
    """Closed-form OLS fit, vectorized across all rows at once.
    The x-grid (1..12) is identical for every row, so a single matrix solve
    replaces thousands of individual sklearn .fit() calls.
    """
    n, T = clipped_12.shape
    x = np.arange(1, T + 1, dtype=float)
    X = np.column_stack([x ** p for p in range(degree + 1)])          # (T, deg+1)
    XtX_inv = np.linalg.pinv(X.T @ X)
    Y = clipped_12.T                                                   # (T, n)
    beta = XtX_inv @ X.T @ Y                                           # (deg+1, n)

    x_full = np.arange(1, T + 2, dtype=float)                          # 13 points
    X_full = np.column_stack([x_full ** p for p in range(degree + 1)])
    pred = X_full @ beta                                                # (13, n)
    return pred.T


def batch_lr(clipped_12):
    return _batch_poly(clipped_12, degree=1)


def batch_pr2(clipped_12):
    return _batch_poly(clipped_12, degree=2)


def batch_pr3(clipped_12):
    return _batch_poly(clipped_12, degree=3)

In [6]:
def batch_ses(clipped_12, alpha=0.8):
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    for t in range(1, T):
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * level[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = clipped_12[:, 0]
    fitted[:, 1:T] = level[:, 0:T - 1]
    fitted[:, T] = level[:, T - 1]
    return fitted


def batch_des(clipped_12, alpha=0.1, beta=0.1):
    n, T = clipped_12.shape
    level = np.zeros((n, T))
    trend = np.zeros((n, T))
    level[:, 0] = clipped_12[:, 0]
    trend[:, 0] = clipped_12[:, 1] - clipped_12[:, 0]
    for t in range(1, T):
        prev_pred = level[:, t - 1] + trend[:, t - 1]
        level[:, t] = alpha * clipped_12[:, t] + (1 - alpha) * prev_pred
        trend[:, t] = beta * (level[:, t] - level[:, t - 1]) + (1 - beta) * trend[:, t - 1]
    fitted = np.zeros((n, T + 1))
    fitted[:, 0] = level[:, 0]
    for t in range(1, T):
        fitted[:, t] = level[:, t - 1] + trend[:, t - 1]
    fitted[:, T] = level[:, T - 1] + trend[:, T - 1]
    return fitted

## 3. Metrics, alerts, and the main pipeline

In [7]:
MODEL_NAMES = ['ma', 'wma', 'ewma', 'lr', 'pr2', 'pr3', 'ses', 'des']


def run_all_models(d_matrix_16, ub, use_fd_window):
    """Fit all 8 models on either the backtest window or the FD window."""
    if use_fd_window:
        actual_12 = np.clip(d_matrix_16[:, -12:], 0, ub[:, None])          # last 12 months
        clipped_15 = np.clip(d_matrix_16[:, -15:], 0, ub[:, None])         # last 15 months
    else:
        actual_12 = np.clip(d_matrix_16[:, -13:-1], 0, ub[:, None])        # 12 months before last
        clipped_15 = np.clip(d_matrix_16[:, :15], 0, ub[:, None])          # first 15 months

    d_last = d_matrix_16[:, -1]

    series = {
        'ma': batch_ma(clipped_15),
        'wma': batch_wma(clipped_15, d_last)[0],
        'ewma': batch_ewma(actual_12),
        'lr': batch_lr(actual_12),
        'pr2': batch_pr2(actual_12),
        'pr3': batch_pr3(actual_12),
        'ses': batch_ses(actual_12),
        'des': batch_des(actual_12),
    }
    return series, actual_12


def compute_metrics(actual_12, series):
    """Vectorized RMSE / MAE / R2 / MAPE / SMAPE per row per model."""
    metrics = {}
    mean_actual = actual_12.mean(axis=1, keepdims=True)
    ss_tot = ((actual_12 - mean_actual) ** 2).sum(axis=1)
    ss_tot_safe = np.where(ss_tot == 0, np.nan, ss_tot)

    for name in MODEL_NAMES:
        pred = series[name][:, :12]
        err = actual_12 - pred
        rmse = np.sqrt((err ** 2).mean(axis=1))
        mae = np.abs(err).mean(axis=1)
        ss_res = (err ** 2).sum(axis=1)
        r2 = 1 - ss_res / ss_tot_safe
        with np.errstate(divide='ignore', invalid='ignore'):
            mape = np.nanmean(np.where(actual_12 == 0, np.nan, np.abs(err) / np.abs(actual_12)), axis=1) * 100
            smape = np.mean(np.abs(err) / (np.abs(actual_12) + np.abs(pred) + 1e-10), axis=1) * 100
        metrics[name] = dict(RMSE=rmse, MAE=mae, R2=r2, MAPE=mape, SMAPE=smape)
    return metrics


def compute_alerts(d_matrix_16):
    last12 = d_matrix_16[:, -12:]
    mean_a = last12.mean(axis=1)
    median_a = np.median(last12, axis=1)
    std_a = last12.std(axis=1)
    max_a = last12.max(axis=1)

    spike = (max_a > np.maximum(mean_a * 1.8, median_a * 1.8)).astype(int)
    with np.errstate(divide='ignore', invalid='ignore'):
        cv = np.where(mean_a != 0, std_a / mean_a, 0)
    volatility = (cv > 1).astype(int)
    zero_count = (last12 == 0).sum(axis=1)
    intermittent = (zero_count >= 6).astype(int)

    score = spike + volatility + intermittent
    label = np.where(score >= 2, 'HIGH RISK', np.where(score == 1, 'REVIEW', 'OK'))
    return spike, volatility, intermittent, score, label

In [8]:
def find_demand_columns(df):
    cols = [c for c in df.columns if re.match(r'^d-\d+$', c)]
    if not cols:
        raise ValueError(
            "Could not find demand columns matching pattern 'd-<number>' "
            "(e.g. d-1, d-2, ... d-16). Check your column headers."
        )
    return sorted(cols, key=lambda x: int(x.split('-')[1]), reverse=True)


def find_branch_column(df):
    for candidate in ('brc', 'branch', 'br'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a branch column (expected 'brc' or 'branch').")


def find_pn_column(df):
    for candidate in ('p/n', 'pn', 'part number'):
        if candidate in df.columns:
            return candidate
    raise ValueError("Could not find a P/N column.")


def find_agc_column(df):
    for candidate in ('agc', 'agency'):
        if candidate in df.columns:
            return candidate
    return None


def forecast(df, include_national_rollup=False):
    """Main entry point. Auto-detects branch/P/N/agc/demand columns.

    When include_national_rollup=True and an agc column is present, two
    extra tiers of rows are added on top of the raw (branch, agc, p/n) rows:
      - branch='ALL', agc=<real agency>   -> that agency's total across all branches
      - branch='ALL', agc='ALL AGC'       -> the grand total across every branch
                                              AND every agency, per P/N
    """
    df = df.copy()
    df.columns = [str(c).strip().replace(chr(10), ' ').lower() for c in df.columns]

    branch_col = find_branch_column(df)
    pn_col = find_pn_column(df)
    agc_col = find_agc_column(df)
    demand_cols = find_demand_columns(df)

    df[pn_col] = df[pn_col].astype(str).str.upper().str.strip()

    keep = [branch_col, pn_col] + ([agc_col] if agc_col else []) + demand_cols
    branch_df = df[keep].rename(columns={branch_col: 'branch', pn_col: 'p/n'})
    if agc_col:
        branch_df = branch_df.rename(columns={agc_col: 'agc'})
    else:
        branch_df['agc'] = np.nan

    frames = [branch_df]
    if include_national_rollup:
        if agc_col:
            # Tier 1: national total PER agency -- sums every branch, keeps agc separate
            national_per_agc = branch_df.groupby(['agc', 'p/n'], as_index=False)[demand_cols].sum()
            national_per_agc.insert(0, 'branch', 'ALL')
            frames.append(national_per_agc[['branch', 'agc', 'p/n'] + demand_cols])

            # Tier 2: grand total across every branch AND every agency for each P/N
            national_all_agc = branch_df.groupby(['p/n'], as_index=False)[demand_cols].sum()
            national_all_agc.insert(0, 'agc', 'ALL AGC')
            national_all_agc.insert(0, 'branch', 'ALL')
            frames.append(national_all_agc[['branch', 'agc', 'p/n'] + demand_cols])
        else:
            national = branch_df.groupby(['p/n'], as_index=False)[demand_cols].sum()
            national.insert(0, 'branch', 'ALL')
            frames.append(national[['branch', 'p/n'] + demand_cols])

    out = pd.concat(frames, ignore_index=True)
    d_matrix = out[demand_cols].to_numpy(dtype=float)
    n = len(out)

    ub_backtest = d_matrix[:, -13:-1].mean(axis=1) + 1.5 * d_matrix[:, -13:-1].std(axis=1)
    ub_fd = d_matrix[:, -12:].mean(axis=1) + 1.5 * d_matrix[:, -12:].std(axis=1)

    series_bt, actual_bt = run_all_models(d_matrix, ub_backtest, use_fd_window=False)
    metrics_bt = compute_metrics(actual_bt, series_bt)
    mae_matrix = np.column_stack([metrics_bt[m]['MAE'] for m in MODEL_NAMES])
    best_idx = np.nanargmin(mae_matrix, axis=1)
    best_model = np.array(MODEL_NAMES)[best_idx]

    series_fd, actual_fd = run_all_models(d_matrix, ub_fd, use_fd_window=True)
    metrics_fd = compute_metrics(actual_fd, series_fd)

    result = pd.DataFrame({
        'branch': out['branch'],
        'agc': out['agc'],
        'p/n': out['p/n'],
    })
    result['clipped_d_FD'] = list(actual_fd)
    for name in MODEL_NAMES:
        result[f'{name}_FD'] = list(series_fd[name])

    result['best_model'] = best_model

    fd_forecast = np.array([series_fd[best_model[i]][i, -1] for i in range(n)])
    result['FD_forecast'] = fd_forecast
    result['FD_final'] = np.maximum(0, np.round(fd_forecast)).astype(int)

    result['metrics_FD'] = [
        [
            {'model': f'{m}_FD', **{k: metrics_fd[m][k][i] for k in ['RMSE', 'MAE', 'R2', 'MAPE', 'SMAPE']}}
            for m in MODEL_NAMES
        ]
        for i in range(n)
    ]
    best_r2_fd = np.array([metrics_fd[best_model[i]]['R2'][i] for i in range(n)])
    result['best_r2_FD'] = best_r2_fd
    result['r2_status_FD'] = np.where(best_r2_fd < 0.25, 'R2 < 0.25', 'Good')

    spike, volatility, intermittent, score, label = compute_alerts(d_matrix)
    result['spike_alert'] = spike
    result['volatility_alert'] = volatility
    result['intermittent_alert'] = intermittent
    result['forecast_alert_score'] = score
    result['forecast_alert_label'] = label

    return result

## 4. Run it

Set `INPUT_FILE` to your current `pmovdcE` export each time you run this. `SHEET_NAME` and `SKIP_ROWS` below are set for a multi-agency CNH export (`pmovdcE_CNH_2_Jul_26.xlsx`, sheet "All", header on row 5, i.e. `skiprows=4`) -- change them to match whatever file you're using, or set both to `None` to auto-detect instead.


In [9]:
INPUT_FILE = "pmovdcE_CNH_2_Jul_26.xlsx"   # <- change this to your current export
SHEET_NAME = "All"                          # e.g. "All", "CNH Steron", "CNH Non Steron" -- or None to auto-detect
SKIP_ROWS = 4                               # rows before the header row -- or None to auto-detect
INCLUDE_NATIONAL_ROLLUP = False   # set False if you only want branch-level rows

t0 = time.time()
raw = load_movement_data(INPUT_FILE, sheet_name=SHEET_NAME, skiprows=SKIP_ROWS)

t1 = time.time()
result = forecast(raw, include_national_rollup=INCLUDE_NATIONAL_ROLLUP)
print(f"Forecast computed for {len(result)} rows in {time.time()-t1:.1f}s")
if result['agc'].notna().any():
    # sort key as str() since agc can mix real agency codes (int) with the
    # 'ALL AGC' grand-total label (str) once national rollup is included
    agencies = sorted(result['agc'].dropna().unique().tolist(), key=str)
    print(f"Agencies in this file ({len(agencies)}): {agencies}")


Loaded: pmovdcE agc 23 3Jul26.xlsx
  sheet: 'pmovdcE agc 23 3Jul26', header row: 5, rows: 31868


C:\Users\Brandon\AppData\Local\Temp\ipykernel_17328\146209268.py:43: RuntimeWarning: Mean of empty slice
  mape = np.nanmean(np.where(actual_12 == 0, np.nan, np.abs(err) / np.abs(actual_12)), axis=1) * 100


Forecast computed for 31868 rows in 1.6s


In [10]:
os.makedirs("output", exist_ok=True)
filename = f"output/forecast_branch_{time.strftime('%Y-%m-%d')}.xlsx"
result.to_excel(filename, index=False)
print(f"Saved to {filename}  ({os.path.getsize(filename)/1e6:.1f} MB)")
print(f"Total runtime: {time.time()-t0:.1f}s")

result.head()

Saved to output/forecast_branch_2026-07-10.xlsx  (5.2 MB)
Total runtime: 59.8s


,branch,agc,p/n,clipped_d_FD,ma_FD,wma_FD,ewma_FD,lr_FD,pr2_FD,pr3_FD,...,FD_forecast,FD_final,metrics_FD,best_r2_FD,r2_status_FD,spike_alert,volatility_alert,intermittent_alert,forecast_alert_score,forecast_alert_label
0,20,23,142784 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
1,20,23,15/25B,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
2,20,23,153518 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
3,20,23,153520 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
4,20,23,154272 S,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,0.0,0,"[{'model': 'ma_FD', 'RMSE': 0.0, 'MAE': 0.0, '...",NaN,Good,0,0,1,1,REVIEW
